# Where do blade-pressure and acoustic errors come from?

This notebook localizes the difference between `p_surface_pred` and `p_surface_true` in `10X7E_7000_surface_time_prediction.npz`.

It separates four questions that should not be conflated:

1. Where is the blade-fixed mean pressure bias?
2. Where is the genuinely time-varying pressure error?
3. Which small parts of the blade contain most of the area-weighted pressure-error energy?
4. Which regions contribute coherently to observer acoustic error?

The fast acoustic section uses a compact rotating-dipole approximation and preserves complex harmonic phase. The final section performs exact regional attribution with the moving, impermeable Farassat 1A solver. In that exact calculation, the pressure-error loading trace is used directly; the geometry-only thickness term is deliberately ignored because it cancels between true and predicted pressure on the same moving mesh.

In [ ]:
from pathlib import Path
from time import perf_counter
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from foamacoustics import F1ASolver, pressure_to_spl_spectrum
from foamacoustics.io.surface import SurfaceFrame
from foamacoustics.plotting import (
    plot_blade_metric,
    plot_blade_pressure_snapshot,
    plot_error_concentration,
    plot_face_pressure_histories,
    plot_harmonic_levels,
    plot_pressure_traces,
    plot_signed_contributions,
    plot_spectrum_comparison,
)

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.22,
        "image.cmap": "viridis",
    }
)

In [ ]:
# Run from the FoamAcoustics repository root.
repo_dir = Path.cwd().resolve()
if not (repo_dir / "pyproject.toml").is_file():
    raise FileNotFoundError(
        "Run this notebook from the FoamAcoustics repository root, "
        "or update repo_dir/data_dir below."
    )

data_dir = repo_dir.parent
archive_path = data_dir / "10X7E_7000_surface_time_prediction.npz"
if not archive_path.is_file():
    raise FileNotFoundError(archive_path)

# Blade/signal diagnostics.
blade_count = 2
snapshot_revolution = 9
snapshot_phase_deg = 0.0
snapshot_blade = 0
harmonic_orders = np.arange(1, 11, dtype=np.int64)
proxy_rotations = 10
radial_edges = np.array([0.0, 0.4, 0.6, 0.8, 0.9, 1.01])
edge_normal_threshold = 0.05

# Observer used by both the compact proxy and optional exact F1A calculation.
observer_m = np.array([1.88, 0.0, 0.0], dtype=np.float64)
sound_speed_m_s = 343.0
reference_pressure_pa = 20.0e-6

# The complete exact analysis is enabled for this one-observer case. Set
# either flag to False for a faster exploratory run; regional attribution
# repeats the 32.7-million-sample moving-surface solve for each region.
run_exact_acoustics = True
run_regional_f1a = True
exact_spectrum_rotations = 8
response_delay_s = 1.0e-5
solver_chunk_size = 64
device = torch.device("cpu")

print(f"Archive: {archive_path}")
print(f"Observer: {observer_m.tolist()} m")
print(f"Torch device: {device}")
print(f"Run exact F1A: {run_exact_acoustics}")
print(f"Run regional F1A: {run_regional_f1a}")

## 1. Load and validate the paired surface-pressure archive

Both arrays live on the same faces and time samples, so their difference is aligned exactly. The stored values are dimensional kinematic-gauge pressure in m²/s². Multiplication by `rho` converts them to pascals; the documented model nondimensionalization is not applied again.

In [ ]:
required_keys = {
    "surface_geometry_ref",
    "surface_geometry_columns",
    "face_id",
    "time",
    "revolutions",
    "p_surface_true",
    "p_surface_pred",
    "rho",
    "rpm",
    "omega",
    "blade_radius",
    "pressure_units",
    "rotation_axis",
    "rotation_origin",
}

with np.load(archive_path, allow_pickle=False) as archive:
    missing = required_keys.difference(archive.files)
    if missing:
        raise KeyError(f"Archive is missing required fields: {sorted(missing)}")

    geometry = np.array(archive["surface_geometry_ref"], copy=True)
    geometry_columns = tuple(archive["surface_geometry_columns"].tolist())
    face_ids = np.array(archive["face_id"], dtype=np.int64, copy=True)
    time_s = np.array(archive["time"], dtype=np.float64, copy=True)
    revolutions = np.array(archive["revolutions"], dtype=np.float64, copy=True)
    p_true_native = np.array(archive["p_surface_true"], dtype=np.float32, copy=True)
    p_pred_native = np.array(archive["p_surface_pred"], dtype=np.float32, copy=True)
    rho_kg_m3 = float(archive["rho"])
    rpm = float(archive["rpm"])
    omega_rad_s = float(archive["omega"])
    blade_radius_m = float(archive["blade_radius"])
    pressure_units = str(archive["pressure_units"].item())
    rotation_axis = np.array(archive["rotation_axis"], dtype=np.float64, copy=True)
    rotation_origin_m = np.array(
        archive["rotation_origin"], dtype=np.float64, copy=True
    )

if p_true_native.shape != p_pred_native.shape:
    raise ValueError("True and predicted pressure arrays are not aligned.")
if p_true_native.shape != (time_s.size, face_ids.size):
    raise ValueError("Pressure shape does not match the time/face metadata.")
if np.any(np.diff(time_s) <= 0.0):
    raise ValueError("Time samples must be strictly increasing.")
if np.unique(face_ids).size != face_ids.size:
    raise ValueError("face_id values must be unique.")

rotation_axis /= np.linalg.norm(rotation_axis)
normalized_units = pressure_units.lower().replace(" ", "").replace("^", "")
if "kinematic" in normalized_units or "m2/s2" in normalized_units:
    native_to_pa = rho_kg_m3
elif normalized_units.startswith("pa"):
    native_to_pa = 1.0
else:
    raise ValueError(f"Unrecognized pressure units: {pressure_units!r}")

# One float32 difference array keeps the default analysis below about 0.4 GB.
p_error_pa = np.subtract(p_pred_native, p_true_native, dtype=np.float32)
p_error_pa *= np.float32(native_to_pa)

rotation_frequency_hz = rpm / 60.0
blade_passing_frequency_hz = blade_count * rotation_frequency_hz
samples_per_revolution = int(
    round((time_s.size - 1) / (revolutions[-1] - revolutions[0]))
)
complete_revolutions = int(round(revolutions[-1] - revolutions[0]))
sampling_rate_hz = 1.0 / np.median(np.diff(time_s))

if time_s.size != complete_revolutions * samples_per_revolution + 1:
    raise ValueError("Expected an integer number of rotations plus one endpoint.")

print(f"Frames/faces: {time_s.size:,} / {face_ids.size:,}")
print(f"Rotations: {complete_revolutions} at {samples_per_revolution} samples/rev")
print(f"Sampling rate: {sampling_rate_hz:,.3f} Hz")
print(f"Shaft/BPF: {rotation_frequency_hz:.6f} / {blade_passing_frequency_hz:.6f} Hz")
print(f"Pressure conversion: {native_to_pa:g} Pa per native unit")
print(
    f"Pressure arrays in memory: {(p_true_native.nbytes + p_pred_native.nbytes + p_error_pa.nbytes) / 2**20:.1f} MiB"
)

## 2. Construct blade-local coordinates

The rotor axis comes from the archive. A weighted principal direction in the rotor plane identifies the two opposite blade spans. Each blade is then mirrored into the same local convention:

- `r/R`: distance from the rotation axis, normalized by blade radius;
- local chord: tangential coordinate with the two blades aligned;
- axial-normal sign: `normal · rotation_axis`, used as a geometric surface label.

The axial-normal labels are intentionally not called pressure/suction side because that convention has not been established independently.

In [ ]:
column_index = {name: index for index, name in enumerate(geometry_columns)}
required_geometry_columns = (
    "x",
    "y",
    "z",
    "normal_x",
    "normal_y",
    "normal_z",
    "area",
)
missing_columns = [
    name for name in required_geometry_columns if name not in column_index
]
if missing_columns:
    raise KeyError(f"Missing geometry columns: {missing_columns}")

centers_m = geometry[
    :, [column_index["x"], column_index["y"], column_index["z"]]
].astype(np.float64)
normals = geometry[
    :,
    [column_index["normal_x"], column_index["normal_y"], column_index["normal_z"]],
].astype(np.float64)
face_area_m2 = geometry[:, column_index["area"]].astype(np.float64)
normals /= np.linalg.norm(normals, axis=1, keepdims=True)

relative_m = centers_m - rotation_origin_m[None, :]
axial_coordinate_m = relative_m @ rotation_axis
rotor_plane_position_m = (
    relative_m - axial_coordinate_m[:, None] * rotation_axis[None, :]
)
weighted_covariance = (
    (rotor_plane_position_m * face_area_m2[:, None]).T @ rotor_plane_position_m
) / face_area_m2.sum()
eigenvalues, eigenvectors = np.linalg.eigh(weighted_covariance)
span_axis = eigenvectors[:, np.argmax(eigenvalues)]
dominant_component = int(np.argmax(np.abs(span_axis)))
if span_axis[dominant_component] < 0.0:
    span_axis *= -1.0
span_axis -= np.dot(span_axis, rotation_axis) * rotation_axis
span_axis /= np.linalg.norm(span_axis)
tangent_axis = np.cross(rotation_axis, span_axis)
tangent_axis /= np.linalg.norm(tangent_axis)

signed_span_m = rotor_plane_position_m @ span_axis
blade_id = (signed_span_m < 0.0).astype(np.int64)
blade_sign = np.where(blade_id == 0, 1.0, -1.0)
local_span_m = blade_sign * signed_span_m
local_chord_m = blade_sign * (rotor_plane_position_m @ tangent_axis)
radius_m = np.linalg.norm(rotor_plane_position_m, axis=1)
radius_over_R = radius_m / blade_radius_m
normal_axis_component = normals @ rotation_axis
# Binary sign is useful for non-overlapping face-center maps. A separate
# three-way label keeps near-edge faces out of ±axis regional conclusions.
surface_side = (normal_axis_component >= 0.0).astype(np.int64)
surface_region = np.where(
    normal_axis_component < -edge_normal_threshold,
    0,
    np.where(normal_axis_component > edge_normal_threshold, 2, 1),
).astype(np.int64)

if np.any(radius_over_R > 1.02):
    raise ValueError("Some face centers lie unexpectedly outside the blade radius.")

print(f"Rotation axis: {rotation_axis}")
print(f"Detected span axis: {span_axis}")
print(f"Detected tangent axis: {tangent_axis}")
for blade in range(blade_count):
    print(f"Blade {blade}: {(blade_id == blade).sum():,} faces")
print(f"Total surface area: {face_area_m2.sum():.8f} m²")
print(
    f"Face-center radius range: {radius_over_R.min():.4f} to {radius_over_R.max():.4f} R"
)

## 3. Per-face pressure-error metrics

The final sample repeats the ten-revolution endpoint, so time statistics and harmonic calculations use the first 3,600 samples. The total face RMSE is decomposed into:

$$\mathrm{MSE}=\overline{e}^{\,2}+\overline{(e-\overline{e})^2},\qquad e=p_{pred}-p_{true}.$$

This matters acoustically: a nonuniform mean error in the blade-fixed frame becomes a rotating load and can generate strong shaft harmonics.

In [ ]:
analysis_slice = slice(0, -1)
error_core_pa = p_error_pa[analysis_slice]
sample_count = error_core_pa.shape[0]

face_bias_pa = error_core_pa.mean(axis=0, dtype=np.float64)
face_mse_pa2 = np.mean(
    np.square(error_core_pa, dtype=np.float32), axis=0, dtype=np.float64
)
face_rmse_pa = np.sqrt(face_mse_pa2)
face_fluctuation_mse_pa2 = np.maximum(face_mse_pa2 - face_bias_pa**2, 0.0)
face_fluctuation_rmse_pa = np.sqrt(face_fluctuation_mse_pa2)
face_mae_pa = np.mean(np.abs(error_core_pa), axis=0, dtype=np.float64)
face_max_abs_error_pa = np.max(np.abs(error_core_pa), axis=0)

# Backward-difference error in dp/dt, accumulated in time chunks to avoid a
# second full time-by-face allocation.
derivative_sum_squares = np.zeros(face_ids.size, dtype=np.float64)
derivative_sample_count = 0
derivative_chunk_size = 240
for start in range(1, sample_count, derivative_chunk_size):
    stop = min(start + derivative_chunk_size, sample_count)
    delta_t_s = time_s[start:stop] - time_s[start - 1 : stop - 1]
    derivative_chunk = (
        error_core_pa[start:stop] - error_core_pa[start - 1 : stop - 1]
    ) / delta_t_s[:, None]
    derivative_sum_squares += np.sum(
        derivative_chunk * derivative_chunk, axis=0, dtype=np.float64
    )
    derivative_sample_count += derivative_chunk.shape[0]
face_derivative_rmse_pa_s = np.sqrt(derivative_sum_squares / derivative_sample_count)

# Rotor-phase repeatability: reshape ten complete revolutions without copying.
error_by_revolution = error_core_pa.reshape(
    complete_revolutions, samples_per_revolution, face_ids.size
)
phase_coherent_error_pa = error_by_revolution.mean(axis=0, dtype=np.float32)
face_phase_coherent_mse_pa2 = np.mean(
    np.square(phase_coherent_error_pa, dtype=np.float32),
    axis=0,
    dtype=np.float64,
)
face_cycle_variable_mse_pa2 = np.maximum(
    face_mse_pa2 - face_phase_coherent_mse_pa2, 0.0
)

total_area = face_area_m2.sum()
area_weighted_error_integral = face_area_m2 * face_mse_pa2
total_weighted_error = area_weighted_error_integral.sum()
area_weighted_rmse_pa = math.sqrt(total_weighted_error / total_area)
area_weighted_bias_rms_pa = math.sqrt(
    np.sum(face_area_m2 * face_bias_pa**2) / total_area
)
area_weighted_fluctuation_rmse_pa = math.sqrt(
    np.sum(face_area_m2 * face_fluctuation_mse_pa2) / total_area
)
bias_mse_fraction = np.sum(face_area_m2 * face_bias_pa**2) / total_weighted_error
phase_coherent_fraction = (
    np.sum(face_area_m2 * face_phase_coherent_mse_pa2) / total_weighted_error
)

print("Global pressure-error summary")
print(f"  Flattened bias: {float(error_core_pa.mean(dtype=np.float64)):.3f} Pa")
print(
    f"  Flattened MAE: {float(np.mean(np.abs(error_core_pa), dtype=np.float64)):.3f} Pa"
)
print(
    f"  Flattened RMSE: {float(np.sqrt(np.mean(np.square(error_core_pa, dtype=np.float32), dtype=np.float64))):.3f} Pa"
)
print(f"  Area-weighted RMSE: {area_weighted_rmse_pa:.3f} Pa")
print(f"  Area-weighted face-bias RMS: {area_weighted_bias_rms_pa:.3f} Pa")
print(f"  Area-weighted fluctuation RMSE: {area_weighted_fluctuation_rmse_pa:.3f} Pa")
print(
    f"  Fraction of weighted MSE from facewise bias: {100.0 * bias_mse_fraction:.2f}%"
)
print(
    f"  Fraction of weighted MSE coherent from revolution to revolution: {100.0 * phase_coherent_fraction:.2f}%"
)
print("  Face-RMSE percentiles [Pa]:")
for percentile in (50, 90, 95, 99, 100):
    print(f"    p{percentile:>3}: {np.percentile(face_rmse_pa, percentile):.3f}")

In [ ]:
side_names = ("normal·axis < 0", "normal·axis ≥ 0")
region_side_names = {
    0: f"−axis-facing (n·axis < −{edge_normal_threshold:g})",
    1: f"edge (|n·axis| ≤ {edge_normal_threshold:g})",
    2: f"+axis-facing (n·axis > {edge_normal_threshold:g})",
}


def make_region_masks(edges=radial_edges):
    regions = []
    for lower, upper in zip(edges[:-1], edges[1:], strict=True):
        radial_mask = (radius_over_R >= lower) & (radius_over_R < upper)
        for side in (0, 1, 2):
            mask = radial_mask & (surface_region == side)
            label = f"{lower:.2f}–{upper:.2f} R, {region_side_names[side]}"
            if np.any(mask):
                regions.append((label, mask))
    membership = np.sum([mask.astype(np.int8) for _, mask in regions], axis=0)
    if not np.all(membership == 1):
        raise ValueError("Regional masks must form a disjoint partition of all faces.")
    return regions


region_masks = make_region_masks()

print("Radial pressure-error summary")
print(
    f"{'r/R':<12} {'area':>9} {'RMSE':>11} {'bias RMS':>11} {'fluct RMS':>11} {'error share':>13}"
)
for lower, upper in zip(radial_edges[:-1], radial_edges[1:], strict=True):
    mask = (radius_over_R >= lower) & (radius_over_R < upper)
    region_area = face_area_m2[mask].sum()
    region_error = np.sum(face_area_m2[mask] * face_mse_pa2[mask])
    region_rmse = math.sqrt(region_error / region_area)
    region_bias_rms = math.sqrt(
        np.sum(face_area_m2[mask] * face_bias_pa[mask] ** 2) / region_area
    )
    region_fluctuation_rms = math.sqrt(
        np.sum(face_area_m2[mask] * face_fluctuation_mse_pa2[mask]) / region_area
    )
    print(
        f"{lower:.2f}–{upper:.2f}   "
        f"{100.0 * region_area / total_area:8.2f}% "
        f"{region_rmse:10.2f} "
        f"{region_bias_rms:10.2f} "
        f"{region_fluctuation_rms:10.2f} "
        f"{100.0 * region_error / total_weighted_error:12.2f}%"
    )

print("\nSurface-side pressure-error summary")
for side in (0, 1, 2):
    mask = surface_region == side
    region_area = face_area_m2[mask].sum()
    region_error = np.sum(face_area_m2[mask] * face_mse_pa2[mask])
    print(
        f"  {region_side_names[side]:<36}: "
        f"area={100.0 * region_area / total_area:6.2f}%, "
        f"RMSE={math.sqrt(region_error / region_area):8.2f} Pa, "
        f"error share={100.0 * region_error / total_weighted_error:6.2f}%"
    )

## 4. Blade maps

The maps use face centers because the compact archive contains face adjacency but not polygon vertices/triangles. Each row is one axial-normal side and each column is one blade, mirrored into a common span/chord convention.

In [ ]:
metric_plots = (
    (
        face_bias_pa,
        "Time-mean pressure error: predicted − true",
        "Mean error [Pa]",
        "coolwarm",
        True,
    ),
    (
        face_fluctuation_rmse_pa,
        "Time-varying pressure-error RMSE after removing each face mean",
        "Fluctuating RMSE [Pa]",
        "magma",
        False,
    ),
    (
        face_rmse_pa,
        "Total pressure-error RMSE",
        "RMSE [Pa]",
        "magma",
        False,
    ),
    (
        face_derivative_rmse_pa_s / 1.0e3,
        "Pressure-error time-derivative RMSE",
        "RMSE(dp error/dt) [kPa/s]",
        "inferno",
        False,
    ),
)
for values, title, colorbar_label, cmap, symmetric in metric_plots:
    plot_blade_metric(
        values,
        radius_over_r=radius_over_R,
        local_chord_m=local_chord_m,
        surface_side=surface_side,
        blade_id=blade_id,
        side_names=side_names,
        title=title,
        colorbar_label=colorbar_label,
        cmap=cmap,
        symmetric=symmetric,
    )
    plt.show()

In [ ]:
# Phase-resolved true/predicted/error map for one blade.
target_revolution = snapshot_revolution + snapshot_phase_deg / 360.0
snapshot_index = int(np.argmin(np.abs(revolutions - target_revolution)))
true_snapshot_pa = native_to_pa * p_true_native[snapshot_index].astype(np.float64)
pred_snapshot_pa = native_to_pa * p_pred_native[snapshot_index].astype(np.float64)

plot_blade_pressure_snapshot(
    true_snapshot_pa,
    pred_snapshot_pa,
    radius_over_r=radius_over_R,
    local_chord_m=local_chord_m,
    surface_side=surface_side,
    blade_id=blade_id,
    selected_blade=snapshot_blade,
    side_names=side_names,
    title=(
        f"Blade {snapshot_blade} at revolution {revolutions[snapshot_index]:.3f} "
        f"(requested phase {snapshot_phase_deg:.1f}°)"
    ),
)
plt.show()

## 5. Error concentration and representative face histories

The Pareto score is the surface integral contribution `area × time-MSE`. It is a mesh-consistent pressure-error concentration measure, not an acoustic power decomposition. The exact acoustic decomposition later preserves coherent phase and cancellation.

In [ ]:
ranked_faces = np.argsort(area_weighted_error_integral)[::-1]
cumulative_area = np.cumsum(face_area_m2[ranked_faces]) / total_area
cumulative_error = (
    np.cumsum(area_weighted_error_integral[ranked_faces]) / total_weighted_error
)
cumulative_face_fraction = np.arange(1, face_ids.size + 1) / face_ids.size

plot_error_concentration(
    cumulative_face_fraction,
    cumulative_area,
    cumulative_error,
)
plt.show()

top_count = 8
print(f"Top {top_count} faces by area × time-MSE")
print(
    f"{'face_id':>8} {'blade':>6} {'r/R':>7} {'chord mm':>10} {'n·axis':>9} {'bias Pa':>10} {'RMSE Pa':>10} {'fluct Pa':>10}"
)
for face_index in ranked_faces[:top_count]:
    print(
        f"{face_ids[face_index]:8d} "
        f"{blade_id[face_index]:6d} "
        f"{radius_over_R[face_index]:7.3f} "
        f"{1.0e3 * local_chord_m[face_index]:10.3f} "
        f"{normal_axis_component[face_index]:9.3f} "
        f"{face_bias_pa[face_index]:10.2f} "
        f"{face_rmse_pa[face_index]:10.2f} "
        f"{face_fluctuation_rmse_pa[face_index]:10.2f}"
    )

In [ ]:
# Inspect true and predicted histories on the most important individual faces.
history_faces = ranked_faces[:4]
history_start = (complete_revolutions - 1) * samples_per_revolution
history_stop = complete_revolutions * samples_per_revolution
history_phase_deg = 360.0 * (
    revolutions[history_start:history_stop] - np.floor(revolutions[history_start])
)
history_titles = [
    f"face {face_ids[index]}, blade {blade_id[index]}, "
    f"r/R={radius_over_R[index]:.3f}, n·axis={normal_axis_component[index]:.3f}"
    for index in history_faces
]
plot_face_pressure_histories(
    history_phase_deg,
    native_to_pa * p_true_native[history_start:history_stop, history_faces],
    native_to_pa * p_pred_native[history_start:history_stop, history_faces],
    face_titles=history_titles,
)
plt.show()

## 6. Fast observer-weighted harmonic attribution

Before running the full retarded-time solver, a compact rotating-dipole model gives a fast, phase-aware answer. For observer direction $\hat{r}$ and distance $r$, each face contributes the scalar rotating load

$$q_f(t)=A_f\,p_f(t)\,[R(t)n_f]\cdot\hat{r}.$$

At frequency $\omega_h$, its approximate acoustic phasor is

$$P_f(\omega_h)\approx\frac{1}{4\pi}\left(\frac{i\omega_h}{c_0 r}+\frac{1}{r^2}\right)Q_f(\omega_h).$$

This approximation treats the rotor as compact and omits the exact moving-surface Doppler and facewise retardation terms. It is useful for screening because complex face/region phasors add exactly and retain constructive or destructive interference. The optional F1A section below is the rigorous check.

In [ ]:
def compact_loading_harmonics(
    pressure_values,
    pressure_scale,
    orders,
    *,
    rotations_to_use,
    chunk_size=240,
):
    orders = np.asarray(orders, dtype=np.int64)
    end = complete_revolutions * samples_per_revolution
    start = end - rotations_to_use * samples_per_revolution
    if start < 0:
        raise ValueError("Requested more complete rotations than are available.")

    observer_relative_m = observer_m - rotation_origin_m
    observer_distance_m = np.linalg.norm(observer_relative_m)
    observer_direction = observer_relative_m / observer_distance_m
    coefficient = np.zeros((orders.size, face_ids.size), dtype=np.complex128)

    axis_cross_observer = np.cross(rotation_axis, observer_direction)
    axis_dot_observer = float(np.dot(rotation_axis, observer_direction))

    for chunk_start in range(start, end, chunk_size):
        chunk_stop = min(chunk_start + chunk_size, end)
        chunk_time_s = time_s[chunk_start:chunk_stop]
        angle = omega_rad_s * (chunk_time_s - time_s[0])
        cosine = np.cos(angle)[:, None]
        sine = np.sin(angle)[:, None]

        # (R n)·observer = n·(R^-1 observer), evaluated without storing a
        # time-by-face-by-vector array.
        observer_in_body = (
            observer_direction[None, :] * cosine
            - axis_cross_observer[None, :] * sine
            + rotation_axis[None, :] * axis_dot_observer * (1.0 - cosine)
        )
        normal_projection = observer_in_body @ normals.T
        pressure_chunk_pa = (
            np.asarray(pressure_values[chunk_start:chunk_stop], dtype=np.float64)
            * pressure_scale
        )
        scalar_loading_n = pressure_chunk_pa * normal_projection * face_area_m2[None, :]
        harmonic_kernel = np.exp(
            -2.0j * np.pi * revolutions[chunk_start:chunk_stop, None] * orders[None, :]
        )
        coefficient += harmonic_kernel.T @ scalar_loading_n

    coefficient *= 2.0 / (end - start)
    frequency_hz = orders * rotation_frequency_hz
    acoustic_factor = (
        1.0j * 2.0 * np.pi * frequency_hz / (sound_speed_m_s * observer_distance_m)
        + 1.0 / observer_distance_m**2
    ) / (4.0 * np.pi)
    acoustic_face_phasor_pa = acoustic_factor[:, None] * coefficient
    return frequency_hz, coefficient, acoustic_face_phasor_pa


proxy_start = perf_counter()
proxy_frequency_hz, true_force_phasor_n, true_acoustic_face_pa = (
    compact_loading_harmonics(
        p_true_native,
        native_to_pa,
        harmonic_orders,
        rotations_to_use=proxy_rotations,
    )
)
_, error_force_phasor_n, error_acoustic_face_pa = compact_loading_harmonics(
    p_error_pa,
    1.0,
    harmonic_orders,
    rotations_to_use=proxy_rotations,
)
pred_acoustic_face_pa = true_acoustic_face_pa + error_acoustic_face_pa
proxy_elapsed_s = perf_counter() - proxy_start

true_acoustic_phasor_pa = true_acoustic_face_pa.sum(axis=1)
pred_acoustic_phasor_pa = pred_acoustic_face_pa.sum(axis=1)
error_acoustic_phasor_pa = error_acoustic_face_pa.sum(axis=1)


def phasor_spl_db(phasor_pa):
    rms_pa = np.maximum(np.abs(phasor_pa) / math.sqrt(2.0), np.finfo(float).tiny)
    return 20.0 * np.log10(rms_pa / reference_pressure_pa)


plot_harmonic_levels(
    harmonic_orders,
    {
        "true": phasor_spl_db(true_acoustic_phasor_pa),
        "predicted": phasor_spl_db(pred_acoustic_phasor_pa),
        "predicted − true phasor": phasor_spl_db(error_acoustic_phasor_pa),
    },
    blade_count=blade_count,
    title=f"Compact rotating-dipole proxy at observer {observer_m.tolist()} m",
    colors={
        "true": "black",
        "predicted": "tab:blue",
        "predicted − true phasor": "tab:red",
    },
)
plt.show()
print(f"Compact harmonic attribution wall time: {proxy_elapsed_s:.2f} s")

In [ ]:
# Coherent regional attribution at the first blade-passing harmonic.
bpf_index = int(np.where(harmonic_orders == blade_count)[0][0])
total_bpf_error_phasor = error_acoustic_phasor_pa[bpf_index]
bpf_denominator = abs(total_bpf_error_phasor) ** 2
if bpf_denominator == 0.0:
    raise RuntimeError(
        "The compact BPF error phasor is zero; attribution is undefined."
    )

proxy_region_rows = []
for label, mask in region_masks:
    region_phasor = error_acoustic_face_pa[bpf_index, mask].sum()
    coherent_share = (
        np.real(region_phasor * np.conj(total_bpf_error_phasor)) / bpf_denominator
    )
    pressure_error_share = (
        np.sum(area_weighted_error_integral[mask]) / total_weighted_error
    )
    region_area_share = face_area_m2[mask].sum() / total_area
    standalone_spl = phasor_spl_db(region_phasor)
    proxy_region_rows.append(
        (label, region_area_share, pressure_error_share, coherent_share, standalone_spl)
    )

print(f"Compact proxy attribution at BPF = {blade_passing_frequency_hz:.3f} Hz")
print(
    f"{'Region':<34} {'area':>8} {'pressure err':>13} {'coherent acoustic':>19} {'standalone':>12}"
)
for (
    label,
    area_share,
    pressure_share,
    coherent_share,
    standalone_spl,
) in proxy_region_rows:
    print(
        f"{label:<34} "
        f"{100.0 * area_share:7.2f}% "
        f"{100.0 * pressure_share:12.2f}% "
        f"{100.0 * coherent_share:18.2f}% "
        f"{standalone_spl:10.2f} dB"
    )
print(f"Coherent shares sum to {100.0 * sum(row[3] for row in proxy_region_rows):.6f}%")
print("Negative or >100% values are valid and expose phase cancellation.")

labels = [row[0].replace(", ", "\n") for row in proxy_region_rows]
shares = 100.0 * np.array([row[3] for row in proxy_region_rows])
plot_signed_contributions(
    labels,
    {"Where the compact-proxy BPF acoustic error comes from": shares},
    ylabel="Signed coherent contribution to BPF error [%]",
    label_rotation_deg=35.0,
    figsize=(11.0, 4.8),
)
plt.show()

face_bpf_coherent_share = (
    np.real(error_acoustic_face_pa[bpf_index] * np.conj(total_bpf_error_phasor))
    / bpf_denominator
)
plot_blade_metric(
    100.0 * face_bpf_coherent_share,
    radius_over_r=radius_over_R,
    local_chord_m=local_chord_m,
    surface_side=surface_side,
    blade_id=blade_id,
    side_names=side_names,
    title="Facewise coherent contribution to compact-proxy BPF acoustic error",
    colorbar_label="Signed contribution [% of total phasor energy projection]",
    cmap="coolwarm",
    symmetric=True,
    percentile=99.0,
)
plt.show()

## 7. Exact moving-surface Farassat 1A attribution

For an impermeable surface on identical moving geometry, Farassat loading is linear in pressure while thickness is pressure-independent. Therefore

$$p_{pred}^{obs}-p_{true}^{obs}=p_{loading}^{obs}[p_{pred}-p_{true}].$$

The notebook-local series below streams either the true pressure or a masked pressure error without creating temporary NPZ files. When the source pressure is an error field, use **only** `loading_pressure_pa`; its `total_pressure_pa` also contains the ordinary rotating-geometry thickness signal and is not an error trace.

The exact true/error comparison and regional solves are enabled in the configuration cell. Set either flag to `False` for a faster exploratory run.

In [ ]:
class ArrayPressureSurfaceSeries:
    """Minimal SurfaceSeries backed by an aligned time-by-face NumPy array."""

    def __init__(self, pressure_values, pressure_scale, face_mask=None):
        if pressure_values.shape != p_true_native.shape:
            raise ValueError("pressure_values must match the archive pressure shape.")
        self.pressure_values = pressure_values
        self.pressure_scale = float(pressure_scale)
        self.face_mask = (
            np.ones(face_ids.size, dtype=np.float64)
            if face_mask is None
            else np.asarray(face_mask, dtype=np.float64)
        )
        if self.face_mask.shape != (face_ids.size,):
            raise ValueError("face_mask must contain one value per face.")

        self.density_kg_m3 = rho_kg_m3
        self.dtype = torch.float64
        self.absolute_times_s = torch.from_numpy(time_s.copy())
        self.source_times_s = self.absolute_times_s - self.absolute_times_s[0]
        self.face_centers_m = torch.from_numpy(centers_m.copy())
        self.face_normals = torch.from_numpy(normals.copy())
        self.face_areas_m2 = torch.from_numpy(face_area_m2.copy())
        self.face_ids = torch.from_numpy(face_ids.copy())
        self.rotation_center_m = torch.from_numpy(rotation_origin_m.copy())
        self.omega_rad_s = torch.from_numpy((omega_rad_s * rotation_axis).copy())
        self.rotation_reference_time_s = float(time_s[0])
        self.rpm = rpm

    def __len__(self):
        return time_s.size

    @property
    def panel_count(self):
        return face_ids.size

    def iter_frames(
        self,
        *,
        workers=1,
        invalid_time_policy="error",
        progress_callback=None,
        progress_every=1,
    ):
        del workers
        if invalid_time_policy not in {"error", "interpolate"}:
            raise ValueError("invalid_time_policy must be 'error' or 'interpolate'.")
        if progress_every < 1:
            raise ValueError("progress_every must be positive.")
        for index in range(len(self)):
            pressure_pa = (
                np.asarray(self.pressure_values[index], dtype=np.float64)
                * self.pressure_scale
                * self.face_mask
            )
            yield SurfaceFrame(
                index=index,
                time_name=f"{time_s[index]:.12g}",
                absolute_time_s=float(time_s[index]),
                source_time_s=float(time_s[index] - time_s[0]),
                pressure_pa=torch.from_numpy(pressure_pa),
                fluid_velocity_m_s=None,
            )
            completed = index + 1
            if progress_callback is not None and (
                completed % progress_every == 0 or completed == len(self)
            ):
                progress_callback(
                    completed,
                    len(self),
                    f"Streamed {completed}/{len(self)} pressure frames",
                )


def solve_array_series(label, series):
    solver = F1ASolver(
        series,
        rpm=rpm,
        permeable=False,
        moving_surface=True,
        rotation_center_m=rotation_origin_m,
        omega_rad_s=omega_rad_s * rotation_axis,
        density_kg_m3=rho_kg_m3,
        sound_speed_m_s=sound_speed_m_s,
        response_delay_s=response_delay_s,
        adjust_time_step=True,
        device=device,
        chunk_size=solver_chunk_size,
    )
    progress = tqdm(total=len(series), desc=label, unit="frame")

    def report_progress(done, total, message):
        del total, message
        progress.update(done - progress.n)

    start = perf_counter()
    try:
        result = solver.predict(
            observer_m,
            progress_callback=report_progress,
            progress_every=25,
            remove_mean=False,
            algorithm="auto",
        )
    finally:
        progress.close()
    print(f"{label}: {perf_counter() - start:.2f} s")
    return result


true_result = None
error_result = None
if run_exact_acoustics:
    true_result = solve_array_series(
        "F1A true pressure",
        ArrayPressureSurfaceSeries(p_true_native, native_to_pa),
    )
    error_result = solve_array_series(
        "F1A pressure error",
        ArrayPressureSurfaceSeries(p_error_pa, 1.0),
    )
    assert torch.isfinite(true_result.total_pressure_pa).all()
    assert torch.isfinite(error_result.loading_pressure_pa).all()
    print(
        "Exact pressure-error trace uses error_result.loading_pressure_pa; "
        "error_result.total_pressure_pa is intentionally ignored."
    )
else:
    print("Exact F1A skipped. Set run_exact_acoustics = True to run it.")

In [ ]:
if run_exact_acoustics:
    exact_end = complete_revolutions * samples_per_revolution
    exact_start = exact_end - exact_spectrum_rotations * samples_per_revolution
    maximum_delay_s = max(
        float(torch.as_tensor(true_result.maximum_delay_s).max()),
        float(torch.as_tensor(error_result.maximum_delay_s).max()),
    )
    if float(true_result.observer_times_s[exact_start]) < maximum_delay_s:
        raise RuntimeError(
            "Exact spectrum window begins before all faces have arrived."
        )

    exact_time_s = true_result.observer_times_s.detach().cpu().numpy()
    exact_true_pa = true_result.total_pressure_pa.detach().cpu().numpy()
    exact_error_pa = error_result.loading_pressure_pa.detach().cpu().numpy()
    exact_pred_pa = exact_true_pa + exact_error_pa

    last_rotation = slice(exact_end - samples_per_revolution, exact_end)
    plot_pressure_traces(
        exact_time_s[last_rotation],
        {
            "true": exact_true_pa[last_rotation],
            "predicted reconstructed as true + loading(error)": exact_pred_pa[
                last_rotation
            ],
            "acoustic error": exact_error_pa[last_rotation],
        },
        time_scale=1.0e3,
        xlabel="Observer time [ms]",
        ylabel="Acoustic pressure [Pa]",
        title="Exact F1A last-rotation histories",
        colors={
            "true": "black",
            "predicted reconstructed as true + loading(error)": "tab:blue",
            "acoustic error": "tab:red",
        },
        linewidths={
            "true": 1.0,
            "predicted reconstructed as true + loading(error)": 1.0,
            "acoustic error": 0.9,
        },
    )
    plt.show()

    exact_spectra = {}
    for label, trace in {
        "true": exact_true_pa,
        "predicted": exact_pred_pa,
        "error": exact_error_pa,
    }.items():
        exact_spectra[label] = pressure_to_spl_spectrum(
            trace[exact_start:exact_end],
            sampling_rate_hz=sampling_rate_hz,
            remove_mean=True,
        )

    plot_spectrum_comparison(
        exact_spectra,
        xlim_hz=(None, 5000.0),
        blade_passing_frequency_hz=blade_passing_frequency_hz,
        bpf_label="BPF",
        bpf_color="tab:orange",
        colors={"true": "black", "predicted": "tab:blue", "error": "tab:red"},
        title="Exact F1A spectra",
        legend_location="best",
        figsize=(10.0, 4.8),
    )
    plt.show()
else:
    print("Exact trace/spectrum plots skipped.")

In [ ]:
regional_loading_results = {}
if run_regional_f1a and not run_exact_acoustics:
    raise ValueError("run_regional_f1a=True requires run_exact_acoustics=True.")

if run_regional_f1a:
    for label, mask in region_masks:
        regional_loading_results[label] = (
            solve_array_series(
                f"F1A error: {label}",
                ArrayPressureSurfaceSeries(p_error_pa, 1.0, face_mask=mask),
            )
            .loading_pressure_pa.detach()
            .cpu()
            .numpy()
        )

    regional_sum_pa = np.sum(list(regional_loading_results.values()), axis=0)
    full_error_pa = error_result.loading_pressure_pa.detach().cpu().numpy()
    closure_error_pa = np.max(np.abs(regional_sum_pa - full_error_pa))
    closure_scale_pa = max(np.max(np.abs(full_error_pa)), np.finfo(float).tiny)
    print(
        f"Regional closure: max abs={closure_error_pa:.3e} Pa, "
        f"relative={closure_error_pa / closure_scale_pa:.3e}"
    )
    if not np.allclose(regional_sum_pa, full_error_pa, rtol=2.0e-9, atol=2.0e-11):
        raise AssertionError(
            "Regional loading traces do not close to the full error trace."
        )

    window = slice(exact_start, exact_end)
    total_window = full_error_pa[window]
    total_energy = float(np.dot(total_window, total_window))
    if total_energy <= np.finfo(float).tiny:
        raise RuntimeError(
            "The exact acoustic-error trace is zero; regional energy projection is undefined."
        )
    exact_bpf_kernel = np.exp(
        -2.0j * np.pi * blade_passing_frequency_hz * exact_time_s[window]
    )
    total_bpf = np.dot(exact_bpf_kernel, total_window)
    if abs(total_bpf) <= np.finfo(float).tiny:
        raise RuntimeError(
            "The exact BPF acoustic-error phasor is zero; regional BPF projection is undefined."
        )

    exact_region_rows = []
    for label, trace in regional_loading_results.items():
        region_window = trace[window]
        energy_projection = float(np.dot(region_window, total_window) / total_energy)
        standalone_rms_pa = float(np.sqrt(np.mean(region_window**2)))
        corrected_rms_reduction = 1.0 - float(
            np.sqrt(np.mean((total_window - region_window) ** 2))
            / np.sqrt(np.mean(total_window**2))
        )
        region_bpf = np.dot(exact_bpf_kernel, region_window)
        bpf_projection = float(
            np.real(region_bpf * np.conj(total_bpf)) / abs(total_bpf) ** 2
        )
        exact_region_rows.append(
            (
                label,
                standalone_rms_pa,
                energy_projection,
                corrected_rms_reduction,
                bpf_projection,
            )
        )

    print("Exact regional acoustic-error attribution")
    print(
        f"{'Region':<34} {'standalone RMS':>15} {'energy projection':>19} {'fix RMS change':>16} {'BPF projection':>16}"
    )
    for (
        label,
        standalone_rms,
        energy_projection,
        rms_reduction,
        bpf_projection,
    ) in exact_region_rows:
        print(
            f"{label:<34} "
            f"{standalone_rms:14.5f} "
            f"{100.0 * energy_projection:18.2f}% "
            f"{100.0 * rms_reduction:15.2f}% "
            f"{100.0 * bpf_projection:15.2f}%"
        )
    print(
        f"Energy projections sum to {100.0 * sum(row[2] for row in exact_region_rows):.6f}%"
    )
    print(
        f"BPF projections sum to {100.0 * sum(row[4] for row in exact_region_rows):.6f}%"
    )

    labels = [row[0].replace(", ", "\n") for row in exact_region_rows]
    plot_signed_contributions(
        labels,
        {
            "Time-domain acoustic-error energy projection": 100.0
            * np.array([row[2] for row in exact_region_rows]),
            "Exact BPF acoustic-error projection": 100.0
            * np.array([row[4] for row in exact_region_rows]),
        },
        label_rotation_deg=40.0,
        label_fontsize=7.0,
        figsize=(13.0, 4.8),
    )
    plt.show()
else:
    print("Regional F1A skipped. Enable both exact-acoustic flags to run it.")

## Reading the results

- Start with the mean-bias map. A blade-fixed bias is not acoustically harmless: rigid rotation turns its nonuniform loading into shaft harmonics.
- Compare total RMSE with fluctuation RMSE. A large difference means the model mainly misses the mean loading distribution rather than transient pressure dynamics.
- Use the Pareto plot to decide where a refined model or additional training data would have the highest leverage.
- Treat standalone regional SPL/RMS as source strength only. It is not additive because regions interfere.
- Use signed coherent projections for attribution. Negative values identify regions that currently cancel part of the total error; correcting one of those regions alone can make the net acoustic error larger.
- The compact proxy is a screening tool. Use exact regional F1A closure and projections before making final acoustic-design conclusions.